# Compare controller settings and fly the course

Prepare two race submissions for the common runner: a baseline using the checked onboard PID/Kalman settings, and an optimized version with a bounded parameter change. Both include your trajectory configuration. Keep the same course and speed for the comparison; predict the effect on lag and overshoot, then compare tracking errors and observed crossings in [02_trajectory.ipynb](02_trajectory.ipynb). Use the existing qualification and race slots.

Use the tested flight tool on the flight laptop. Assign a pilot, separate emergency-stop operator, telemetry monitor and recorder; everyone can call a stop. Keep the area clear. If tuning or hardware is unavailable, use the checked untuned runner or shared recordings. Planning and analysis work offline.


In [ ]:
from pathlib import Path
import sys
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "code/day3/plan.py").is_file())
sys.path.insert(0, str(root / "code"))
from day3.flight import dry_run, fly, new_attempt
from notebook_radio import check_drone
from day3.tuning import capture_baseline, make_tuning, race_tuning, tuning_changes
from day3.notebook_tools import load_plan
from IPython.display import display
run_id = "flight-01"  # Change when starting another full session.
output = root / "code/day3/data" / run_id
output.mkdir(parents=True, exist_ok=False)
print("Notebook Python:", sys.executable)
print("Session files:", output)


## R1: check the example before surveying

After Setup, set `prepare_example=True` below and run the cell. It plans the synthetic example and writes matched `course.json` and `summary.json` files into a new folder.

Then go directly to **Check the course and tuning together**. Enter the runner paths from the run card, set `course_json=example_course_json` and `run_dry_check=True`, and leave `submission_to_check=None`. This check uses no radio or tuning capture. Keep the example flight-disabled; it checks the software interface, not the room geometry. For R4, select your exported measured course or submission instead.


In [ ]:
from trajectory_tools.course import read_json
from day3.notebook_tools import plan_course, save_plan
prepare_example = False
example_course_json = checked_run = None
if prepare_example:
    tuning_spec = None
    example_source = root / "examples/course.synthetic.json"
    example_plan = plan_course(read_json(example_source), provenance={
        "source_template": "examples/course.synthetic.json", "geometry_status": "synthetic",
    })
    example_directory = new_attempt(output / "example-checks")
    save_plan(example_plan, example_directory)
    example_course_json = example_directory / "course.json"
    print("Example course and summary:", example_directory)


## 1. Identify the grounded drone

Place the drone on a stable surface, keep clear of the propellers and close other radio clients. Set `LABEL` to the sticker label (hexadecimal 0-B), run **Connect** and confirm the blinking drone. This identifies the drone and leaves firmware stop engaged.


In [ ]:
LABEL = None  # For example "1", "0" or "B".
uri = None
if LABEL is not None:
    uri = check_drone(LABEL)
else:
    print("Set LABEL when preparing a flight.")


## Capture the baseline

With the drone grounded and firmware stop engaged, set `capture_tuning_baseline=True` once. The helper verifies the stop and reads the active PID controller (1), Kalman estimator (2), horizontal position P gains and available horizontal acceleration process noise. No tuning is applied. Keep the resulting baseline for this drone throughout the comparison; a file for another URI is rejected.

If controller/estimator settings differ, check them in cfclient's Parameters tab (`stabilizer.controller=1`, `stabilizer.estimator=2`), power-cycle and check stable baseline hover before returning here. An unstable baseline is a reason to use the fallback, rather than tune around a hardware or positioning fault.

To reuse a completed capture, set `baseline_path` to its JSON and leave capture disabled. The capture leaves firmware stop engaged.


In [ ]:
capture_tuning_baseline = False
baseline_path = None  # Or Path("/absolute/path/to/baseline.json") for this drone.
tuning_spec = checked_run = None
if capture_tuning_baseline:
    if uri is None:
        raise ValueError("Identify the grounded drone first.")
    baseline_directory = new_attempt(output / "baselines")
    baseline_path = capture_baseline(uri, baseline_directory)
print("Tuning baseline:", baseline_path)


## 2. Choose one change

**Core controller comparison.** First use `mode="baseline"`, `factor=1.0`, `factor_range=None` and `range_checked=False`. For the second attempt choose `mode="controller"` and a factor different from 1: the factor scales `posCtlPid.xKp` and `posCtlPid.yKp` together, preserving their ratio. Keep geometry, speed, estimator and inner-loop settings fixed. Predict whether tracking lag, overshoot and successful clearance will improve.

Agree a narrow `factor_range=[low, high]` within the software ceiling **0.9-1.1**, including 1, and cross-check the choice against Day 1's results and the baseline hover. Set `range_checked=True` only after that discussion. This ceiling is a software guard, not a demonstrated safe range for every airframe. Start the changed configuration at low speed; oscillation or poor tracking means stop and return to baseline. Leave attitude gains and safety limits unchanged.

**Optional Kalman comparison.** Declare the KF profile before the **12:30 estimator freeze**; qualify and compare it in an allocated flight slot afterward. This can replace another trial if time allows; use shared results if no slot remains. With `mode="kalman"`, scale `kalman.pNAcc_xy` within an agreed range inside **0.5-2.0**, keeping controller gains at baseline. Predict the Day 2 responsiveness/noise trade-off, then compare estimate jitter, tracking and recovery from naturally missing measurements. Do not deliberately obscure Lighthouse during flight. Day 2's fitted values need a model/unit mapping before transfer.

The helper applies one family only, checks fresh readbacks, and restores the original values after landing or abort, without persisting them. A failed restoration keeps the attempt unsuccessful: ground the drone and verify the recorded baseline before continuing.


In [ ]:
mode = "baseline"  # "controller" for the core comparison; "kalman" is optional.
parameter_group = None  # Core defaults; optional stretch choices are listed below.
factor = 1.0
factor_range = None  # Candidate: the group's agreed [low, high], containing 1.
range_checked = False
tuning_spec = tuning_choices = race_configuration = checked_run = None
if baseline_path is not None:
    if mode == "baseline":
        tuning_spec = make_tuning(baseline_path, mode=mode, factor=factor,
                                  factor_range=factor_range, range_checked=range_checked,
                                  parameter_group=parameter_group)
    else:
        group = parameter_group or ("position_p_xy" if mode == "controller" else "acceleration_xy")
        tuning_choices = {mode: {group: factor}, "factor_range": factor_range,
                          "range_checked": range_checked}
        tuning_spec = race_tuning(baseline_path, **tuning_choices)
    for change in tuning_changes(tuning_spec):
        requested_value = change["baseline"] * change["factor"]
        print(f"{change['parameter']}: {change['baseline']:g} * {change['factor']:g} = {requested_value:g}")
    if mode == "baseline":
        print("Baseline selected: captured controller/KF values remain unchanged.")
elif mode != "baseline" or parameter_group is not None:
    raise ValueError("Capture or select this drone's baseline before tuning.")
else:
    print("Untuned fallback selected; no experimental parameter changes.")


### Stretch: compare another loop or noise contribution

Choose any stretch profile during planning and declare it before the relevant freeze: **12:30 for KF, 13:00 for controller settings**. Qualify and compare it later in an allocated flight slot, replacing another trial rather than extending the session. Keep the same course and speed, predict its effect, compare against the captured baseline, and use the same agreed factor interval and low-speed check.

Set `parameter_group` in the cell above (leave it `None` for the core defaults):

| `mode` | `parameter_group` | Firmware parameters and question |
| --- | --- | --- |
| `controller` | `position_i_xy` / `position_d_xy` | `posCtlPid.xKi/yKi` or `xKd/yKd`: does integral action reduce sustained error, or derivative action change damping and noise sensitivity? |
| `controller` | `velocity_p_xy` / `velocity_i_xy` / `velocity_d_xy` | `velCtlPid.vxKp/vyKp`, `vxKi/vyKi`, or `vxKd/vyKd`: how does the inner velocity loop affect position tracking? |
| `kalman` | `acceleration_z` | `kalman.pNAcc_z`: how does vertical responsiveness trade against altitude jitter? |
| `kalman` | `velocity` / `position` | `kalman.pNVel` or `kalman.pNPos`: how does additional process uncertainty affect tracking and recovery? |

Each trial scales one group; paired X/Y gains preserve their ratio. Software bounds remain 0.9-1.1 for PID and 0.5-2.0 for KF, with a narrower group-agreed interval. A zero baseline stays zero under multiplication, so the helper rejects that choice. Inspect the capture and choose an available positive-valued group. A fresh capture includes all available groups; an older capture may need replacing.

KF factors scale individual firmware fields, whose contributions to covariance depend on the firmware model and time step. Compare the observed response rather than interpreting a factor as scaling the entire covariance matrix. Keep attitude gains, safety limits and all other groups at baseline. The runner checks and restores stretch parameters through the same acknowledgement and audit path as the core comparison.


## Export the two race submissions

Choose a bounded controller change above, then select the course saved by the trajectory notebook. Keep its `course.json` and `summary.json` together: the API checks they match, retains the survey and clearance-budget evidence, and recomputes the trajectory. It shows the path, limits and exact parameter changes. Set `export_submissions=True` to write `baseline/` and `optimized/`, each with `course.json`, `tuning.json`, sampled trajectory and summary. The baseline checks the captured values; the optimized profile declares the temporary change. Start from the agreed checked baseline before tuning. Optional KF tuning uses `mode="kalman"`.

Hand over both folders for the common flight laptop and runner. Each needs its own dry run and allocated low-speed check. Export keeps flight disabled; enabling flight or changing any setting requires another dry run.

To check a handed-over submission, set `submission_to_check` below to its `baseline/` or `optimized/` folder. The cell loads the course and profile together. Keep `summary.json` alongside the course for the dry check and flight; enabling flight is the only course change allowed by that summary. Other changes need a new planner export. Then run the dry check and preflight. A baseline for another drone is rejected. The runner archives the course, profile and provenance summary and verifies parameter restoration for each attempt.


In [ ]:
from race_common.tuning import load_tuning
export_submissions = False
planned_course = None  # Path to course.json exported by 02_trajectory.ipynb.
race_configuration = submissions = checked_run = None
if planned_course is not None and tuning_choices is not None:
    selected_plan = load_plan(planned_course)
    race_configuration = selected_plan.configure_race(baseline=baseline_path, **tuning_choices)
    display(race_configuration.summary())
    figure = selected_plan.plot()
    display(figure)
    import matplotlib.pyplot as plt
    plt.close(figure)
if export_submissions:
    if race_configuration is None:
        raise ValueError("Select your planned course and bounded optimized tuning first.")
    submission_folder = new_attempt(output / "submissions")
    submissions = race_configuration.export(submission_folder / "race")
    display(submissions)


## Check the course and tuning together

Enter the runner file, Python interpreter and working directory from the run card, and your exported course or submission folder. Inspect the trajectory preview first. The dry run checks geometry, limits and support for the shared version-1 reference, including gate speeds, crossing offsets and turning waypoint velocities. The runner rounds total duration up by less than one command period.

The planner exports `enabled_for_flight=false`. After checking the course, edit that field to `true` in Jupyter's text editor, save, then repeat the dry run. Any subsequent course, tuning, runner or interpreter change requires another dry run. Keep the tool installation and dependencies unchanged between dry run and flight.

Each check gets a new numbered folder under `checks/`, including failed checks. After editing or enabling a course, rerun the next cell with `run_dry_check=True`; earlier reports stay available.

The tested runner must support the explicit tuning option; unsupported tools fail the dry run before any flight. Before each freeze deadline, dry-run the baseline and each planned candidate, and record their `.tuning.json` paths in the run card. This declares the exact baseline, mode, parameter group, factor and interval. Later qualification and comparison select only these declared settings. Changing any setting requires another dry run and is allowed only before its deadline.


In [ ]:
runner = None  # Path("/absolute/path/to/flight_runner.py")
runner_python = None  # Path("/absolute/path/to/qualified/.venv/bin/python")
runner_directory = None  # Path("/absolute/path/to/flight-tool")
course_json = None  # R1: example_course_json; R4: Path to your exported course.json.
submission_to_check = None  # Path to the exported baseline/ or optimized/ folder.
run_dry_check = False

checked_run = None
if run_dry_check:
    if submission_to_check is not None:
        folder = Path(submission_to_check)
        course_json = folder / "course.json"
        tuning_spec = load_tuning(folder / "tuning.json", uri)
    if any(value is None for value in (runner, runner_python, runner_directory, course_json)):
        raise ValueError("Set the four flight-tool and course paths first.")
    check_directory = new_attempt(output / "checks")
    print("Dry-run report:", check_directory / "console.txt")
    checked_run = dry_run(runner, course_json, runner_python, runner_directory,
                          check_directory / "console.txt", tuning=tuning_spec)
else:
    print("Dry run not selected; planning and recording analysis work offline.")


## 3. Power-cycle, check and fly

**After Connect** and baseline capture, place the drone on the ground, switch it fully off, then on, and let it finish starting up. This clears firmware stop. Keep the discovered `uri` and proceed here without rerunning Connect, which would engage the stop again.

Cross-check the drone and course together: supports and floor changes, cage clearance, Lighthouse visibility, battery, measured mass and controller, flight volume, tracking limit, landing and stop path. Complete the [race preflight checks](../../docs/race_practical.md#r5-qualify-and-freeze-the-race-configuration), with the separate emergency-stop operator ready. Repeat these checks after handling or rebooting.

When the group has completed the power cycle and preflight, set `ready_to_fly=True` and run the next cell. Type the exact course name when asked. The runner also checks this response. Each launch gets a new numbered folder under `attempts/`, containing its course snapshot, provenance summary (`race.summary.json`), tuning file, parameter/readback audit, telemetry and console report. The summary retains the survey and clearance budget, with hashes of the exact flown course and tuning. These records remain after landing or abort. The runner reads that snapshot; later planning changes leave this attempt's geometry intact.

Interrupting the cell requests the runner's controlled landing and waits for it to finish. Keep the kernel and laptop running during landing. If the drone behaves unsafely, use the agreed emergency stop immediately; a software landing is best-effort.

For another attempt, power-cycle and repeat preflight, then rerun the launch cell with explicit readiness. Repeat the dry run first if the course, tuning or flight tool has changed. After landing or an abort, use the printed telemetry and archived course paths in the trajectory notebook's recording-analysis cell.

Use the same baseline capture for the paired attempts. Compare RMS/maximum error, lag/overshoot and observed crossings, with battery condition noted. Record one prediction and whether it held in the existing team notes. Changing speed is a later comparison after choosing stable gains, not part of the matched gain test.

Freeze the set of comparison profiles by the programme deadlines. Use R5's baseline check as attempt A. The first R6 flight can be the changed-gain low-speed qualification (B); use the remaining slot with the better validated profile. Select only predeclared profiles after the freeze, and treat an unqualified configuration as a test rather than a serious race attempt. At a fixed time-based reference, improved tracking mainly affects clearance and successful completion; it does not automatically shorten the programmed duration.


In [ ]:
ready_to_fly = False
telemetry = flown_course_json = None
if ready_to_fly:
    ready_to_fly = False
    attempt = new_attempt(output / "attempts")
    telemetry = attempt / "race-telemetry.csv"
    flown_course_json = telemetry.with_suffix(".course.json")
    console = attempt / "console.txt"
    print("Attempt files:", attempt)
    fly(checked_run, uri, telemetry, console, ready=True, tuning=tuning_spec)
else:
    print("Flight not selected.")
